# 🚀 Scraper NowGoal Optimizado (Google Colab)

Este notebook te permite ejecutar el scraper de NowGoal en la nube usando los servidores de Google.

**Pasos:**
1. Primero sube el archivo **`project_code.zip`** (generado con `preparar_kit_colab.bat`) en la sección de **Archivos** (icono carpeta izquierda) o ejecutando la celda de Carga.
2. Ejecuta las celdas en orden.

In [ ]:
#@title 1. Instalación de Dependencias
print("Instalando librerías necesarias...")
!pip install requests beautifulsoup4 pandas colorama pytz python-dotenv user_agent
print("✅ Instalación completada.")

In [ ]:
#@title 2. Cargar/Descomprimir Código
import zipfile
import os
import shutil
from google.colab import files

# Opción para subir archivo si no se ha subido manual
if not os.path.exists('project_code.zip'):
    print("⬆️ Sube el archivo 'project_code.zip'...")
    uploaded = files.upload()

if os.path.exists('project_code.zip'):
    # Limpiar entorno anterior por seguridad
    if os.path.exists('src'): shutil.rmtree('src')
    if os.path.exists('recopilacion_data'): shutil.rmtree('recopilacion_data')
    
    print("📦 Descomprimiendo archivos...")
    with zipfile.ZipFile('project_code.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    
    # --- REPARACIÓN DE RUTAS WINDOWS ---
    # En algunos casos, el zip de Windows crea archivos llamados "carpeta\archivo.py" 
    # en lugar de ponerlos dentro de carpetas. Esto lo arregla.
    print("🔧 Verificando y reparando estructura de archivos...")
    repaired_count = 0
    for filename in os.listdir('.'):
        if '\\' in filename:
            # Es un archivo con ruta de windows incrustada en el nombre
            real_path = filename.replace('\\', '/')
            directory = os.path.dirname(real_path)
            
            # Crear directorio si no existe
            if directory and not os.path.exists(directory):
                os.makedirs(directory)
            
            # Mover archivo
            shutil.move(filename, real_path)
            repaired_count += 1
            
    if repaired_count > 0:
        print(f"✅ Se repararon {repaired_count} archivos con rutas de Windows.")
    else:
        print("✅ Estructura correcta.")

    
    # Verificación final de estructura
    if os.path.exists('recopilacion_data') and os.path.exists('src'):
        print("✅ TODO LISTO: Código cargado correctamente.")
    else:
        print("⚠️ ADVERTENCIA: Algo sigue raro. Listando archivos raíz:")
        print(os.listdir('.'))
    
    # Crear carpeta data si no existe
    if not os.path.exists('data'):
        os.makedirs('data')
else:
    print("❌ No se encontró 'project_code.zip'. Súbelo primero.")

---

In [ ]:
#@title 🔎 Diagnóstico de Conexión (Opcional)
#@markdown Ejecuta esto solo si obtienes "0 partidos encontrados". Comprueba si NowGoal bloquea la IP de Colab.
import sys
import os
# FIX: Añadir 'src' al path para que funcionen los imports relativos
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('src'))

import requests
from src.modules.estudio_scraper import get_requests_session_of

print("Probando conexión con NowGoal...")
try:
    session = get_requests_session_of()
    # Usamos headers simulando navegador real
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
    }
    target_url = "https://www.nowgoal6.com/"
    r = session.get(target_url, headers=headers, timeout=15)
    
    print(f"Status Code: {r.status_code}")
    print(f"Content Length: {len(r.text)} bytes")
    
    if r.status_code == 200:
        if "match" in r.text.lower() or "game" in r.text.lower():
            print("✅ CONEXIÓN EXITOSA: Parece que vemos contenido correctamente.")
        else:
            print("⚠️ ALERTA: Recibimos 200 OK pero el contenido parece vacío o bloqueado.")
            print("Primeros 500 caracteres:")
            print(r.text[:500])
    else:
        print(f"❌ ERROR DE HTTP: El servidor devolvió {r.status_code}.")
except Exception as e:
    print(f"❌ ERROR CRÍTICO DE CONEXIÓN: {e}")

In [ ]:
#@title 3a. Cachear Partidos Terminados (Página Principal)
#@markdown Scrapea partidos terminados recientes y los guarda directamente en los archivos finales `data_ah_*.json`.

WORKERS = 15 #@param {type:"slider", min:5, max:30, step:1}
HANDICAP = "all" #@param ["all", "0", "0.25", "0.5", "0.75", "1", "1.25", "1.5"]
GOAL_LINE = "all" #@param ["all", "2", "2.25", "2.5", "2.75", "3"]

import sys
import os
sys.path.append('.')

script_path = 'recopilacion_data/wrapper_cachear_terminados.py'

if os.path.exists(script_path):
    print(f"🚀 Iniciando Cacheo de Terminados con {WORKERS} workers...")
    !python {script_path} {HANDICAP} {GOAL_LINE} {WORKERS}
else:
    print(f"❌ ERROR: No encuentro el archivo '{script_path}'")
    print("Archivos en directorio actual:", os.listdir('.'))

In [ ]:
#@title 3b. Precacheo (Buscar Partidos Nuevos)
#@markdown Busca partidos PRÓXIMOS (o pendientes) y los añade a `data_precacheo.json`.

WORKERS_PRE = 10 #@param {type:"slider", min:5, max:20, step:1}

import sys
import os
sys.path.append('.')

script_path = 'recopilacion_data/wrapper_scrapear_pendientes.py'

if os.path.exists(script_path):
    print(f"🚀 Iniciando Precacheo con {WORKERS_PRE} workers...")
    !python {script_path} {WORKERS_PRE}
else:
    print(f"❌ ERROR: No encuentro el archivo '{script_path}'")
    if os.path.exists('recopilacion_data'):
        print("Contenido de recopilacion_data:", os.listdir('recopilacion_data'))
    else:
        print("No existe la carpeta recopilacion_data")
    print("Archivos en directorio actual:", os.listdir('.'))

In [ ]:
#@title 3c. Buscar Resultados Pendientes y Finalizar
#@markdown Ejecuta el ciclo completo: Busca resultados (+2h) y Mueve al explorador.

print("🚀 1. Buscando resultados...")
!python recopilacion_data/wrapper_buscar_resultados.py
print("\n🚀 2. Finalizando partidos...")
!python recopilacion_data/wrapper_finalizar_todos.py

---

In [ ]:
#@title 4. Descargar Resultados (ZIP)
#@markdown Empaqueta toda la carpeta `data/` y la descarga.

from google.colab import files
import shutil
import datetime
import os

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
base_name = f"resultados_nowgoal_{timestamp}"

print("📦 Comprimiendo carpeta data/ ...")
shutil.make_archive(base_name, 'zip', 'data')

zip_file = f"{base_name}.zip"
print(f"⬇️ Descargando {zip_file} ...")
files.download(zip_file)